# Module 5 — Deploying ML models (Telco churn): commented walkthrough

Alexey's lecture code (lessons 5.2 – 5.6) plus the 2025 **workshop** that replaces its tools, with comments that explain **three things** for every step:

- `# GOAL`   what this step is for, in plain words
- `# WHY`    the engineering / ML logic behind it (what could go wrong if we skipped it)
- `# PYTHON` how the Python / scikit-learn / Flask / FastAPI syntax works, so you can write it yourself later

(Dockerfiles, Pipfile and pyproject.toml use `# SYNTAX` instead of `# PYTHON`.)

The original, uncommented notebook and scripts are untouched in `../notebooks/` (`code/` = lecture, `workshop/` = uv + FastAPI workshop).

**How this notebook is organised.** Module 5 is mostly *files*, not notebook cells. The commented copies live in `commented/` (`train.py`, `predict.py`, `predict-test.py`, `ping.py`, `Pipfile`, `Dockerfile`, and `workshop/train.py`, `workshop/predict.py`, `workshop/pyproject.toml`, `workshop/Dockerfile`). Read each file in the editor when its section says so. The notebook runs the Python parts for real: it trains and pickles the model, then **starts each web service as a background process**, sends it requests and stops it again, so you see the whole request → prediction → response loop without juggling terminals. Run it from `practice/`.

Needs, in the same environment as Jupyter: `flask`, `gunicorn`, `fastapi`, `uvicorn`, `requests` (`pip install flask gunicorn fastapi uvicorn requests` inside the `ml-zoomcamp` env).

Differences from the lecture:
- the data is read from the local file (`data-week-3.csv`, the same Telco file as Modules 3 – 4);
- text columns are found with `select_dtypes`. On pandas 3 the lecture's `df.dtypes == 'object'` selects **nothing**, every churn label becomes 0 and `LogisticRegression` stops with *"only one class"*. Same fix in `commented/train.py` and `commented/workshop/train.py`;
- `!ls -lh *.bin` is replaced by a Python file-size check;
- the lecture starts the service in a separate terminal and the "Making requests" cells assume it is running. Here helper functions start/stop it (Flask in 5.3, gunicorn in 5.4, uvicorn in the workshop part);
- the workshop's `make_pipeline(DictVectorizer(), LogisticRegression(solver='liblinear'))` crashes on **scikit-learn 1.9** at `fit` (*"Only sparse matrices with 32-bit integer indices are accepted"*: 1.9's DictVectorizer emits 64-bit sparse indices, liblinear refuses them). Here and in `commented/workshop/train.py` it is `DictVectorizer(sparse=False)`. HW5 is not affected: it only *predicts* with the frozen pipeline, inside the locked scikit-learn;
- added: the **workshop** version (one sklearn `Pipeline`, FastAPI, Pydantic validation, uv), because **that is the stack Homework 5 uses**, and a 415/422 demo of what happens when a request is malformed;
- pipenv, uv and Docker commands are terminal commands: they are shown in markdown, not run.

**Not executed when this notebook was checked:** the `docker build` / `docker run` commands (the build container could not pull images from Docker Hub / ghcr.io). Everything else, including the three services and their requests, was executed end to end.

## Lecture vs workshop vs Homework 5: three stacks, one idea

Lessons 5.3 – 5.6 were recorded in 2021. The course README says: watch them for the **theory**, then do the practical part with the **workshop** (`../notebooks/workshop/README.md`, video https://www.youtube.com/watch?v=jzGzw98Eikk). **Homework 5 follows the workshop, not the lessons.**

| | Lessons 5.2 – 5.6 (2021) | Workshop (2025) | **Homework 5 (2026)** |
|---|---|---|---|
| Artifact | `pickle.dump((dv, model))`: a tuple | `pickle.dump(pipeline)`: one `Pipeline` | `pipeline.bin`: one `Pipeline`, frozen, checksummed |
| Predict call | `model.predict_proba(dv.transform([c]))` | `pipeline.predict_proba(c)` | `pipeline.predict_proba([record])` |
| Web framework | Flask, `request.get_json()` | FastAPI + Pydantic model | FastAPI + Pydantic model |
| Server | `gunicorn --bind=0.0.0.0:9696 predict:app` | `uvicorn predict:app --host 0.0.0.0 --port 9696` | `uv run uvicorn predict:app --host 0.0.0.0 --port 9696` |
| Environment | pipenv: `Pipfile` + `Pipfile.lock` | uv: `pyproject.toml` + `uv.lock` | uv, `uv sync --locked` (never re-lock) |
| Python | 3.8 | 3.13 | **3.11.15** (`.python-version`) |
| Docker base | `python:3.8.12-slim` | `python:3.13.5-slim-bookworm` | the tag in `../homework/Dockerfile` |
| Response keys | `churn_probability`, `churn` | `churn_probability`, `churn` | `conversion_probability`, `conversion` |
| Cloud | AWS Elastic Beanstalk (5.7) | Fly.io | none |

The ideas carry over one to one: save an artifact, load it once at start-up, wrap prediction in an HTTP endpoint, freeze the environment, put everything in a container. Only the tool names change.

## 5.1 Intro / Session overview

Until now the model lived inside a notebook. **Deployment** = making it usable by another program: the marketing system sends a customer, our *churn service* answers with a probability. The path this module walks:

notebook → **train script** that saves the model (5.2) → **web service** that loads it and answers HTTP requests (5.3 – 5.4) → **isolated environment** with pinned versions (5.5) → **container** that runs the same everywhere (5.6) → **cloud** (5.7, optional).

Physics analogy: the trained model is a calibrated instrument. Deployment is shipping it with its calibration certificate (pinned versions, checksums) and a standard connector (HTTP + JSON), so it reads the same in someone else's lab.

## 5.2 Saving and loading the model

First the Module 4 model, rebuilt in a few cells (train with 5-fold CV for `C=1.0`, then the final model), then `pickle` to write it to disk and read it back.

In [17]:
# PYTHON: 'import X as Y' loads a library under a short nickname.
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [18]:
# GOAL: load and clean the Telco data exactly as in Modules 3 and 4.
df = pd.read_csv('data-week-3.csv')

df.columns = df.columns.str.lower().str.replace(' ', '_')

# WHY: select_dtypes works on old pandas (text = 'object') AND pandas 3 (text = 'str').
#      The lecture's df.dtypes == 'object' finds no column on pandas 3 -> nothing is cleaned below.
categorical_columns = list(df.select_dtypes(include=['object', 'string']).columns)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

df.churn = (df.churn == 'yes').astype(int)

# Sanity check: ~0.27. If it prints 0.0, the cleaning above did not run.
df.churn.mean()

np.float64(0.2653698707936959)

In [19]:
# GOAL: 80 / 20 split. Model selection happens with K-fold CV inside the 80%; the 20% is the final exam.
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)

In [20]:
numerical = ['tenure', 'monthlycharges', 'totalcharges']

categorical = [
    'gender', 'seniorcitizen', 'partner', 'dependents',
    'phoneservice', 'multiplelines', 'internetservice',
    'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
    'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod',
]

In [21]:
# GOAL: the Module 4 helper pair: dataframe -> fitted (dv, model), and dataframe -> probabilities.
# WHY: the DictVectorizer is created inside train(), so every CV fold fits its own (no leakage).
def train(df_train, y_train, C=1.0):
    dicts = df_train[categorical + numerical].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(dicts)

    # NOTE: expect ConvergenceWarnings in the CV loop below: totalcharges (thousands) and the 0/1 columns
    #       have very different scales, so lbfgs runs out of iterations. The AUC is fine (see Module 4).
    model = LogisticRegression(C=C, max_iter=1000)
    model.fit(X_train, y_train)

    return dv, model


def predict(df, dv, model):
    dicts = df[categorical + numerical].to_dict(orient='records')

    X = dv.transform(dicts)
    y_pred = model.predict_proba(X)[:, 1]

    return y_pred

In [22]:
# GOAL: the run's parameters in one place: these become the top of train.py.
C = 1.0
n_splits = 5

In [23]:
# GOAL: 5-fold CV for C=1.0: mean and spread of the validation AUC.
kfold = KFold(n_splits=n_splits, shuffle=True, random_state=1)

scores = []

for train_idx, val_idx in kfold.split(df_full_train):
    df_train = df_full_train.iloc[train_idx]
    df_val = df_full_train.iloc[val_idx]

    y_train = df_train.churn.values
    y_val = df_val.churn.values

    dv, model = train(df_train, y_train, C=C)
    y_pred = predict(df_val, dv, model)

    auc = roc_auc_score(y_val, y_pred)
    scores.append(auc)

print('C=%s %.3f +- %.3f' % (C, np.mean(scores), np.std(scores)))

/opt/anaconda3/envs/ml-zoomcamp/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/opt/anaconda3/envs/ml-zoomcamp/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/m

C=1.0 0.842 +- 0.007


/opt/anaconda3/envs/ml-zoomcamp/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [24]:
scores    # the five fold AUCs

[0.8445141336515137,
 0.845138475036419,
 0.8333574152117198,
 0.8347728944170876,
 0.851801010322864]

In [25]:
# GOAL: the FINAL model: retrain on all of df_full_train, score once on the test set.
# WHY: this (dv, model) pair is what we deploy. The test AUC should be close to the CV mean.
dv, model = train(df_full_train, df_full_train.churn.values, C=1.0)
y_pred = predict(df_test, dv, model)

y_test = df_test.churn.values
auc = roc_auc_score(y_test, y_pred)
auc

/opt/anaconda3/envs/ml-zoomcamp/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.8583057081261442

### Save the model

In [26]:
# GOAL: pickle = Python's built-in way to turn (almost) any object into bytes and back.
# WHY: the trained weights live in this kernel's memory; when the kernel stops they are gone.
#      A file on disk is what another process (the web service) can load.
import pickle

In [27]:
# PYTHON: f-string -> 'model_C=1.0.bin'. Putting C in the name records which setting produced the file.
output_file = f'model_C={C}.bin'
output_file

'model_C=1.0.bin'

In [28]:
# GOAL: the long way first: open, write, close.
# PYTHON: 'wb' = write binary. pickle.dump(obj, file) writes obj into the open file.
#         (dv, model) is ONE tuple: both objects travel together, because the service needs both.
# WHY: if you forget f_out.close(), the bytes may still sit in a buffer and the file is incomplete.
f_out = open(output_file, 'wb')
pickle.dump((dv, model), f_out)
f_out.close()

In [29]:
# GOAL: check the file exists and how big it is (the lecture uses !ls -lh *.bin).
# WHY: a logistic regression is just ~45 weights + the dv's column list: a few KB.
from pathlib import Path

for p in Path('.').glob('*.bin'):
    print(p.name, p.stat().st_size, 'bytes')

model_C=1.0.bin 2459 bytes


In [30]:
# GOAL: the idiomatic way. The 'with' block closes the file automatically, even after an error.
with open(output_file, 'wb') as f_out:
    pickle.dump((dv, model), f_out)

### Load the model

Pretend this is a new process that knows nothing about training: restart the kernel and run only the cells from here (plus the `import pickle`). That is exactly the situation of the web service.

In [31]:
import pickle

In [32]:
input_file = 'model_C=1.0.bin'

In [33]:
# PYTHON: 'rb' = read binary. pickle.load returns exactly what was dumped: the tuple, unpacked in one line.
# WHY: scikit-learn must be importable here (pickle stores the class NAMES, not their code), and should
#      be the same version that wrote the file. A different version loads with an
#      InconsistentVersionWarning and is not guaranteed to give the same numbers: that is why HW5
#      runs inside a locked uv environment.
# WHY (security): unpickling can execute code. Only load pickles you produced or trust.
with open(input_file, 'rb') as f_in:
    dv, model = pickle.load(f_in)

In [34]:
model
dv

,"dtype dtype: dtype, default=np.float64The type of feature values. Passed to Numpy array/scipy.sparse matrixconstructors as the dtype argument.",<class 'numpy.float64'>
,"separator separator: str, default=""=""Separator string used when constructing new features for one-hotcoding.",'='
,"sparse sparse: bool, default=TrueWhether transform should produce scipy.sparse matrices.",False
,"sort sort: bool, default=TrueWhether ``feature_names_`` and ``vocabulary_`` should besorted when fitting.",True


In [35]:
# GOAL: one customer, as a dict with the CLEANED training values (lower case, underscores).
customer = {
    'gender': 'female',
    'seniorcitizen': 0,
    'partner': 'yes',
    'dependents': 'no',
    'phoneservice': 'no',
    'multiplelines': 'no_phone_service',
    'internetservice': 'dsl',
    'onlinesecurity': 'no',
    'onlinebackup': 'yes',
    'deviceprotection': 'no',
    'techsupport': 'no',
    'streamingtv': 'no',
    'streamingmovies': 'no',
    'contract': 'month-to-month',
    'paperlessbilling': 'yes',
    'paymentmethod': 'electronic_check',
    'tenure': 1,
    'monthlycharges': 29.85,
    'totalcharges': 29.85
}

In [36]:
# WHY: transform wants a LIST of dicts (one per row): [customer] is a list with one row.
X = dv.transform([customer])
X

array([[ 1.  ,  0.  ,  0.  ,  1.  ,  0.  ,  1.  ,  0.  ,  0.  ,  1.  ,
         0.  ,  1.  ,  0.  ,  0.  , 29.85,  0.  ,  1.  ,  0.  ,  0.  ,
         0.  ,  1.  ,  1.  ,  0.  ,  0.  ,  0.  ,  1.  ,  0.  ,  1.  ,
         0.  ,  0.  ,  1.  ,  0.  ,  1.  ,  0.  ,  0.  ,  1.  ,  0.  ,
         0.  ,  1.  ,  0.  ,  0.  ,  1.  ,  0.  ,  0.  ,  1.  , 29.85]])

In [37]:
# PYTHON: [0, 1] = row 0, column 1 = P(churn).
y_pred = model.predict_proba(X)[0, 1]

In [38]:
print('input:', customer)
print('output:', y_pred)

input: {'gender': 'female', 'seniorcitizen': 0, 'partner': 'yes', 'dependents': 'no', 'phoneservice': 'no', 'multiplelines': 'no_phone_service', 'internetservice': 'dsl', 'onlinesecurity': 'no', 'onlinebackup': 'yes', 'deviceprotection': 'no', 'techsupport': 'no', 'streamingtv': 'no', 'streamingmovies': 'no', 'contract': 'month-to-month', 'paperlessbilling': 'yes', 'paymentmethod': 'electronic_check', 'tenure': 1, 'monthlycharges': 29.85, 'totalcharges': 29.85}
output: 0.628084365611833


### Turning the notebook into a script

Lesson 5.2 ends by exporting this notebook (`jupyter nbconvert --to script`) and cleaning it into `train.py`. Open **`commented/train.py`** now and read it top to bottom: it is the cells above, with the parameters at the top and the `with open(...)` save at the end. Running it re-creates the same file:

In [39]:
# PYTHON: '!' runs a shell command; {sys.executable} is the Python of THIS kernel, so the script
#         sees the same pandas / sklearn as the notebook.
import sys

!{sys.executable} commented/train.py

doing validation with C=1.0
/opt/anaconda3/envs/ml-zoomcamp/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
auc on fold 0 is 0.8445141336515137
/opt/anaconda3/envs/ml-zoomcamp/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to s

## 5.3 Web services: introduction to Flask

A **web service** is a program that listens on a port and answers HTTP requests. The smallest one answers `GET /ping` with `PONG`: read **`commented/ping.py`**.

In a terminal you would run `python commented/ping.py` and then `curl http://localhost:9696/ping` (or open the URL in a browser). Here two helpers start the server as a background process and stop it again.

In [45]:
# GOAL: start a server command in the background, wait until it accepts connections, stop it later.
# WHY: a web service runs "forever"; a notebook cell must finish. A background process (subprocess.Popen)
#      lets the notebook play the CLIENT while the server runs next to it.
import socket
import subprocess
import time

PORT = 9696


def port_in_use(port=PORT):
    # PYTHON: connect_ex returns 0 when something accepts the connection.
    with socket.socket() as s:
        return s.connect_ex(('127.0.0.1', port)) == 0


def start_server(cmd, port=PORT, timeout=30):
    # WHY: if port 9696 is taken (a docker container, a uvicorn left running in a terminal),
    #      the new server would fail and our requests would silently hit the OLD one.
    if port_in_use(port):
        raise RuntimeError(f'port {port} is already in use: stop the other server first (docker ps / Ctrl+C)')
    proc = subprocess.Popen(cmd)
    deadline = time.time() + timeout
    while time.time() < deadline:
        if proc.poll() is not None:
            raise RuntimeError(f'server exited with code {proc.returncode}: run the command in a terminal to see why')
        if port_in_use(port):
            return proc
        time.sleep(0.3)
    proc.terminate()
    raise RuntimeError('server did not start in time')


def stop_server(proc):
    proc.terminate()
    proc.wait(timeout=10)

In [47]:
!{sys.executable} -m pip install flask

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [flask]


In [48]:
# GOAL: run ping.py with Flask's development server.
# WHY: 'python -m flask run' instead of 'python ping.py': the script uses debug=True, whose auto-reloader
#      spawns a child process that a notebook cannot stop cleanly. Same app, same port.
server = start_server([sys.executable, '-m', 'flask', '--app', 'commented/ping.py', 'run', '--port', str(PORT)])

 * Serving Flask app 'commented/ping.py'
 * Debug mode: off


 * Running on http://127.0.0.1:9696
Press CTRL+C to quit


127.0.0.1 - - [18/Oct/2026 18:17:08] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [18/Oct/2026 18:17:08] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [18/Oct/2026 18:17:08] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [18/Oct/2026 18:17:08] "GET /favicon.ico HTTP/1.1" 404 -


In [ ]:
# GOAL: be the client. requests.get = what the browser does when you open the URL.
import requests

r = requests.get(f'http://localhost:{PORT}/ping')
r.status_code, r.text

127.0.0.1 - - [18/Oct/2026 18:17:30] "GET /ping HTTP/1.1" 200 -


(200, 'PONG')

127.0.0.1 - - [18/Oct/2026 18:17:35] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [18/Oct/2026 18:17:37] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [18/Oct/2026 18:17:37] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [18/Oct/2026 18:17:37] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [18/Oct/2026 18:17:37] "GET / HTTP/1.1" 404 -


In [50]:
stop_server(server)

## 5.4 Serving the churn model with Flask

Read **`commented/predict.py`**: the "Load the model" cells above, wrapped in a `POST /predict` endpoint. It loads `model_C=1.0.bin` once at start-up, turns the JSON body into a dict, and returns `churn_probability` and `churn`.

Flask's built-in server prints *"do not use it in a production deployment"*. The lecture's production server is **gunicorn** (`gunicorn --bind=0.0.0.0:9696 predict:app`, where `predict:app` = the variable `app` in `predict.py`). On Windows it would be waitress; on the Mac gunicorn works.

In [ ]:
# GOAL: serve predict.py with gunicorn, exactly as the lecture's Dockerfile does.
# PYTHON: --pythonpath commented lets gunicorn import predict.py from commented/, while the working
#         directory stays practice/, where model_C=1.0.bin is.
server = start_server([sys.executable, '-m', 'gunicorn', f'--bind=127.0.0.1:{PORT}',
                       '--pythonpath', 'commented', 'predict:app'])

### Making requests (the lecture's last cells)

In [ ]:
url = f'http://localhost:{PORT}/predict'

In [ ]:
# WHY: same customer as before, but on a TWO-year contract: expect a much lower churn probability.
customer = {
    'gender': 'female',
    'seniorcitizen': 0,
    'partner': 'yes',
    'dependents': 'no',
    'phoneservice': 'no',
    'multiplelines': 'no_phone_service',
    'internetservice': 'dsl',
    'onlinesecurity': 'no',
    'onlinebackup': 'yes',
    'deviceprotection': 'no',
    'techsupport': 'no',
    'streamingtv': 'no',
    'streamingmovies': 'no',
    'contract': 'two_year',
    'paperlessbilling': 'yes',
    'paymentmethod': 'electronic_check',
    'tenure': 1,
    'monthlycharges': 29.85,
    'totalcharges': 29.85
}

In [ ]:
# PYTHON: json=customer serialises the dict AND sets the header Content-Type: application/json.
response = requests.post(url, json=customer).json()

In [ ]:
response

In [ ]:
# GOAL: the business action the prediction exists for.
if response['churn']:
    print('sending email to', 'asdx-123d')
else:
    print('no email for', 'asdx-123d')

In [ ]:
# GOAL: what goes wrong with a hand-written request. data= sends the same JSON text but WITHOUT the
#       Content-Type header (what 'curl -d' does unless you add -H 'Content-Type: application/json').
# WHY: Flask's get_json() refuses it with 415 Unsupported Media Type. FastAPI (workshop, HW5) answers 422.
import json

bad = requests.post(url, data=json.dumps(customer))
bad.status_code

In [ ]:
# GOAL: the lecture's client script, as a separate program (read commented/predict-test.py).
!{sys.executable} commented/predict-test.py

In [ ]:
stop_server(server)

## Workshop: one Pipeline + FastAPI (the Homework 5 stack)

The workshop redoes 5.2 – 5.4 with modern tools. Two changes matter for the homework:

1. **One artifact.** `make_pipeline(DictVectorizer(), LogisticRegression(...))` glues the two objects together; you pickle and load *one* thing and call `pipeline.predict_proba(...)` on raw dicts. HW5's `pipeline.bin` is exactly this.
2. **FastAPI instead of Flask.** The request body is declared as a Pydantic class, so bad input is rejected with HTTP 422 before it reaches the model, and you get free docs at `/docs`.

Read **`commented/workshop/train.py`** and **`commented/workshop/predict.py`**.

In [ ]:
# GOAL: the workshop's training: a Pipeline fitted on ALL rows (no split: the workshop is about deployment).
# WHY: same features, same cleaning, so we reuse df, categorical and numerical from 5.2.
from sklearn.pipeline import make_pipeline

train_dict = df[categorical + numerical].to_dict(orient='records')
y_full = df.churn

# WHY: sparse=False is a fix for scikit-learn 1.9: its DictVectorizer now emits sparse matrices with
#      64-bit indices and liblinear refuses them ("Only sparse matrices with 32-bit integer indices are
#      accepted"). The workshop's DictVectorizer() worked on 1.7/1.8. Dense = same model, ~45 columns, no issue.
pipeline = make_pipeline(
    DictVectorizer(sparse=False),
    LogisticRegression(solver='liblinear')
)
pipeline.fit(train_dict, y_full)

# PYTHON: named_steps shows the two stages inside the one object.
pipeline.named_steps

In [ ]:
# GOAL: predicting is now ONE call on the raw dict (the DictVectorizer step runs inside).
pipeline.predict_proba(customer)[0, 1]

In [ ]:
# GOAL: save ONE object as model.bin: the file the workshop's predict.py loads.
with open('model.bin', 'wb') as f_out:
    pickle.dump(pipeline, f_out)

In [ ]:
# GOAL: serve it with uvicorn (the ASGI server FastAPI runs on; gunicorn's counterpart).
# PYTHON: --app-dir commented/workshop = where to import predict.py from; model.bin is read from practice/.
server = start_server([sys.executable, '-m', 'uvicorn', '--app-dir', 'commented/workshop', 'predict:app',
                       '--host', '127.0.0.1', '--port', str(PORT)])

In [ ]:
# GOAL: same request as with Flask. Same response keys, because the workshop kept them.
requests.post(url, json=customer).json()

In [ ]:
# GOAL: Pydantic validation at work: an impossible value is refused with 422 and an explanation.
# PYTHON: {**customer, 'streamingtv': 'maybe'} = a copy of the dict with one key overridden.
bad = requests.post(url, json={**customer, 'streamingtv': 'maybe'})
bad.status_code, bad.json()['detail'][0]['msg']

In [ ]:
# GOAL: the free documentation page. While the server runs, open http://localhost:9696/docs in a browser
#       and try /predict from there ("Try it out"). Here we just check it exists.
requests.get(f'http://localhost:{PORT}/docs').status_code

In [ ]:
stop_server(server)

## 5.5 Python virtual environment: Pipenv

**Why:** project A needs scikit-learn 0.24, project B needs 1.7. One global Python cannot hold both, and a pickle must be loaded with the version that wrote it. A *virtual environment* per project solves it; a *lock file* makes it reproducible on another machine.

Read **`commented/Pipfile`** (lecture) and **`commented/workshop/pyproject.toml`** (workshop and HW5). Same job, different tool:

| Task | pipenv (lecture) | **uv (workshop, HW5)** |
|---|---|---|
| start a project | `pipenv install` | `uv init` |
| add a package | `pipenv install flask` | `uv add fastapi` |
| add a dev-only package | `pipenv install --dev requests` | `uv add --dev requests` |
| fresh copy: install everything | `pipenv install` | `uv sync` |
| install exactly the lock, fail if stale | `pipenv install --deploy` | `uv sync --locked` |
| run a command inside the env | `pipenv run gunicorn ...` | `uv run uvicorn ...` |
| files | `Pipfile`, `Pipfile.lock` | `pyproject.toml`, `uv.lock`, `.python-version` |

Install uv on the Mac with `brew install uv` (or `curl -LsSf https://astral.sh/uv/install.sh | sh`). uv downloads the Python version in `.python-version` by itself, so it does not matter that your conda env has a different one.

Commands to try in a terminal, in `../notebooks/workshop/`:

```bash
uv sync --locked                                              # builds .venv/ from uv.lock
uv run uvicorn predict:app --host 0.0.0.0 --port 9696 --reload
uv run python test.py                                         # in a second terminal
```

In [ ]:
# GOAL: read a lock file the way you will read HW5's: which exact version of a package got locked?
# WHY: pyproject.toml says what you ASKED for (here '>=1.7.1'); uv.lock says what was INSTALLED.
# PYTHON: tomllib (standard library, Python 3.11+) parses TOML; uv.lock has a [[package]] list.
import tomllib

with open('../notebooks/workshop/uv.lock', 'rb') as f:
    lock = tomllib.load(f)

{p['name']: p['version'] for p in lock['package'] if p['name'] in ('scikit-learn', 'fastapi', 'uvicorn')}

## 5.6 Environment management: Docker

A virtual environment isolates Python packages; a **container** isolates everything: OS libraries, the Python interpreter, the packages and the code. The image you build on the Mac runs identically on a Linux server.

Read **`commented/Dockerfile`** (lecture: Python 3.8 + pipenv + gunicorn) and **`commented/workshop/Dockerfile`** (uv + uvicorn). Same five moves in both: base image → install the env manager → copy the lock files and install → copy code + model → start the server.

Docker Desktop must be running (whale icon in the menu bar), otherwise every command ends with *"Cannot connect to the Docker daemon"*. In a terminal:

```bash
cd ../notebooks/workshop                     # has model.bin, uv.lock, .python-version, Dockerfile
docker build -t predict-churn .
docker run -it --rm -p 9696:9696 predict-churn
# second terminal:
python test.py                               # or curl, as in 5.4
docker ps                                    # what is running and which ports it holds
```

- `-p 9696:9696` = HOST:CONTAINER. Without it the container listens on 9696 *inside* and nothing on the Mac can reach it (`EXPOSE` in the Dockerfile only documents the port).
- `--rm` deletes the stopped container; `-it` lets Ctrl+C stop it.
- Only one process can hold port 9696 on the Mac. Stop a uvicorn/gunicorn left running in another terminal **before** `docker run`, or the run fails with *"port is already allocated"*, and your requests keep hitting the old local server instead of the container.

The lecture image (`../notebooks/code/`) is 2021-era (Python 3.8, scikit-learn 0.24.2); building it is optional. The image that matters for the homework is `../homework/Dockerfile`.

In [ ]:
# GOAL: check from here whether Docker is installed and its daemon answers (needed from Thu 22 Oct).
import shutil

if shutil.which('docker') is None:
    print('docker not found: install Docker Desktop before the Docker session')
else:
    # PYTHON: subprocess.run runs a command and captures its output; a non-zero returncode means the
    #         client works but the daemon (Docker Desktop) is not running.
    r = subprocess.run(['docker', 'version', '--format', '{{.Server.Version}}'], capture_output=True, text=True)
    print('Docker daemon', r.stdout.strip() if r.returncode == 0 else 'NOT reachable: start Docker Desktop')

## 5.7 Deployment to the cloud: AWS Elastic Beanstalk (optional)

Optional, and you have no AWS account until Nov 10: **watch or skim, don't do**. The idea: `eb init -p docker` + `eb create` takes the same Docker image and runs it on a managed AWS machine with a public URL. The workshop does the same with Fly.io (`fly launch`, `fly deploy`). HW5 has no cloud question. Remember to delete anything you create (`eb terminate`, `fly apps destroy`): idle cloud services still bill.

## 5.8 Summary

| Step | Lecture tool | Homework tool | Watch out |
|---|---|---|---|
| Save the model | `pickle.dump((dv, model))` | `pickle.dump(pipeline)` | load with the **same** scikit-learn version; only unpickle trusted files |
| Script | `train.py` | `train.py` (writes checksums + metadata) | parameters at the top; HW5's frozen `pipeline.bin` must not be retrained |
| Web service | Flask + `request.get_json()` | FastAPI + Pydantic | `Content-Type: application/json`; load the model once, not per request |
| Server | gunicorn | uvicorn | `--host 0.0.0.0` inside a container; port 9696 |
| Environment | pipenv | uv (`uv sync --locked`, `uv run`) | the lock file, not your conda env, defines the versions |
| Container | Dockerfile + `docker run -p` | same, pinned image digests | Docker Desktop running; one process per host port |
| Cloud | Elastic Beanstalk | (Fly.io in the workshop) | delete what you create |

## 5.9 Explore more

- Add `model_config = ConfigDict(extra="forbid")` to the workshop's `Customer` and send an extra key: 422 instead of silence
- Add a `GET /health` endpoint that returns the model file's SHA-256 (HW5's `predict.py` does this: read how)
- Turn `commented/predict.py` into a FastAPI app yourself without looking at the workshop, then diff
- Try other ways to deploy a container: Google Cloud Run, AWS App Runner, Fly.io